# Object Storage Data Lab

MCMP 서버의 broker가 기존 CB-Tumblebug API로 발급한 presigned URL을 제한된 터널을 통해 제공합니다.
Jupyter 컨테이너에는 CSP Access Key / Secret Key와 CB-Tumblebug 자격증명이 없습니다.
노트북은 고정된 broker URL을 사용하며, broker가 만료 전에 새 presigned URL로 자동 교체합니다.

버킷에 파일을 올린 뒤 아래 셀을 위에서부터 실행하세요.

## 1. 버킷 파일 목록

In [ ]:
import os
import matplotlib.pyplot as plt
from object_storage_access import create_connection, list_objects, object_url, sql_identifier, sql_quote, upload_file

con = create_connection()
objects = list_objects()
files = [str(item.get("key", "")) for item in objects if item.get("key")]
print("storage:", os.environ.get("OBJECT_STORAGE_ID", ""))
print("objects:", len(files))
for name in files[:30]:
    print(" -", name)

## 2. 데이터 로드

Parquet이 있으면 Parquet을, 없으면 CSV를 읽습니다. `TARGET_KEYS`에 분석할 객체 키 목록을 직접 지정할 수도 있습니다.

In [ ]:
TARGET_KEYS = None

def pick_reader():
    selected = list(TARGET_KEYS) if TARGET_KEYS else []
    if selected:
        reader = "read_parquet" if all(name.lower().endswith(".parquet") for name in selected) else "read_csv_auto"
        return selected, reader
    parquet_files = [name for name in files if name.lower().endswith(".parquet")]
    if parquet_files:
        return parquet_files, "read_parquet"
    csv_files = [name for name in files if name.lower().endswith(".csv")]
    if csv_files:
        return csv_files, "read_csv_auto"
    return [], None

selected_keys, reader = pick_reader()
if not selected_keys:
    df = None
    print("읽을 Parquet / CSV 파일이 없습니다. 버킷에 파일을 올린 뒤 1번 셀부터 다시 실행하세요.")
else:
    urls = [object_url(name) for name in selected_keys]
    url_list_sql = "[" + ", ".join(sql_quote(url) for url in urls) + "]"
    df = con.execute("SELECT * FROM " + reader + "(" + url_list_sql + ", union_by_name=true)").df()
    print(reader, "objects:", len(selected_keys))
    print("rows:", len(df), "columns:", list(df.columns))
    display(df.head())

## 3. 집계와 차트

문자열 컬럼을 기준으로 숫자 컬럼을 합계 냅니다. `GROUP_COL` / `VALUE_COL`로 직접 지정할 수 있습니다.

In [ ]:
GROUP_COL = None
VALUE_COL = None
summary = None

if df is None or df.empty:
    print("로드된 데이터가 없습니다.")
else:
    text_cols = [c for c in df.columns if df[c].dtype == object]
    num_cols = [c for c in df.columns if df[c].dtype.kind in "ifu"]
    group_col = GROUP_COL or ("region" if "region" in df.columns else (text_cols[0] if text_cols else None))
    value_col = VALUE_COL or (num_cols[0] if num_cols else None)
    if group_col is None or value_col is None:
        print("집계할 컬럼을 찾지 못했습니다. GROUP_COL과 VALUE_COL을 직접 지정하세요.")
    else:
        con.register("loaded", df)
        summary = con.execute(
            "SELECT " + sql_identifier(group_col) + " AS group_key, SUM(" + sql_identifier(value_col) + ") AS total "
            "FROM loaded GROUP BY 1 ORDER BY 1"
        ).df()
        display(summary)
        summary.plot.bar(x="group_key", y="total", legend=False, title=value_col + " by " + group_col)
        plt.tight_layout()
        plt.show()

## 4. 새 파일 추가 후 재실행

버킷에 파일을 더 올린 뒤 1~3번 셀을 다시 실행하면 목록과 결과, 차트가 갱신됩니다.

## 5. 분석 결과 저장 (선택)

In [ ]:
if summary is None:
    print("저장할 집계 결과가 없습니다.")
elif os.environ.get("WRITE_RESULT_ENABLED", "true").lower() != "true":
    print("WRITE_RESULT_ENABLED가 false라 저장하지 않습니다.")
else:
    result_prefix = os.environ.get("RESULT_PREFIX", "results").strip("/")
    result_key = result_prefix + "/summary.parquet"
    local_result = "/tmp/object-storage-data-lab-summary.parquet"
    con.register("summary_data", summary)
    con.execute("COPY summary_data TO " + sql_quote(local_result) + " (FORMAT PARQUET)")
    upload_file(result_key, local_result)
    print("Saved:", result_key)
    display(con.execute("SELECT * FROM read_parquet(" + sql_quote(object_url(result_key)) + ")").df())